In [ ]:
import sys
import os
from pyspark import SparkContext, SparkConf

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

conf = SparkConf().setAppName("TrafficGraphConstruction").setMaster("local[2]")

try:
    sc.stop()
except:
    pass

sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

INPUT_FILE = "file:///home/reza/HW4/camera_passes.csv"

def parse_traffic_line(line):
    try:
        parts = line.split(',')
        if parts[0] == "RecordID" or parts[1] == "CameraID":
            return None
        
        camera_id = parts[1].strip()
        plate_number = parts[2].strip()
        record_time = parts[3].strip()
        
        return (plate_number, (record_time, camera_id))
    except:
        return None

raw_rdd = sc.textFile(INPUT_FILE)
parsed_rdd = raw_rdd.map(parse_traffic_line).filter(lambda x: x is not None)

grouped_rdd = parsed_rdd.groupByKey()

def create_transitions(data):
    plate, sightings = data
    sorted_sightings = sorted(sightings, key=lambda x: x[0])
    
    links = []
    for i in range(len(sorted_sightings) - 1):
        src_cam = sorted_sightings[i][1]
        dst_cam = sorted_sightings[i+1][1]
        
        if src_cam != dst_cam:
            links.append((src_cam, dst_cam))
            
    return links

edges_rdd = grouped_rdd.flatMap(create_transitions)

adjacency_list = edges_rdd.groupByKey().mapValues(list)

total_nodes = edges_rdd.flatMap(lambda x: [x[0], x[1]]).distinct().count()
total_edges = edges_rdd.count()

print("\n" + "="*40)
print("Graph Construction Report")
print("="*40)
print(f"Total Nodes: {total_nodes}")
print(f"Total Edges: {total_edges}")
print("-" * 40)
print("Sample Adjacency List (First 5 nodes):")

for node, neighbors in adjacency_list.take(5):
    print(f"Camera {node} --> {neighbors[:10]} ... (Total: {len(neighbors)})")
print("="*40 + "\n")

In [ ]:
import sys
import os
from operator import add
from pyspark import SparkContext, SparkConf

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

conf = SparkConf().setAppName("TrafficPageRank_Fixed").setMaster("local[2]")

try:
    sc.stop()
except:
    pass

sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

INPUT_FILE = "file:///home/reza/HW4/camera_passes.csv"

def parse_traffic_line(line):
    try:
        parts = line.split(',')
        if parts[0] == "RecordID" or parts[1] == "CameraID":
            return None
        camera_id = parts[1].strip()
        plate_number = parts[2].strip()
        record_time = parts[3].strip()
        if not camera_id: 
            return None
        return (plate_number, (record_time, camera_id))
    except:
        return None

raw_rdd = sc.textFile(INPUT_FILE)
parsed_rdd = raw_rdd.map(parse_traffic_line).filter(lambda x: x is not None)

grouped_rdd = parsed_rdd.groupByKey()

def create_transitions(data):
    plate, sightings = data 
    sorted_sightings = sorted(sightings, key=lambda x: x[0])
    links = []
    for i in range(len(sorted_sightings) - 1):
        src = sorted_sightings[i][1]
        dst = sorted_sightings[i+1][1]
        if src != dst:
            links.append((src, dst))
    return links

edges = grouped_rdd.flatMap(create_transitions)


links = edges.distinct().groupByKey().mapValues(list)
links.cache() 


all_nodes = edges.flatMap(lambda x: [x[0], x[1]]).distinct()
total_nodes_count = all_nodes.count()
if total_nodes_count == 0:
    raise ValueError("The transition graph contains no nodes")

dangling_nodes = all_nodes.subtract(links.map(lambda x: x[0]))
dangling_count = dangling_nodes.count()
full_links = links.union(dangling_nodes.map(lambda x: (x, [])))
full_links.cache() 

ranks = all_nodes.map(lambda x: (x, 1.0 / total_nodes_count))

ITERATIONS = 10
BETA = 0.85

print(f"\nTotal Nodes: {total_nodes_count}")
print(f"Dangling Nodes: {dangling_count}")
print("Starting PageRank Iterations...\n")

all_nodes_pair = all_nodes.map(lambda x: (x, None))
all_nodes_pair.cache()

for i in range(ITERATIONS):
    dangling_mass = full_links.join(ranks).filter(lambda x: len(x[1][0]) == 0).map(lambda x: x[1][1]).sum()
    base_probability = ((1 - BETA) + BETA * dangling_mass) / total_nodes_count
    contribs = full_links.join(ranks).flatMap(
        lambda x: [(dest, x[1][1] / len(x[1][0])) for dest in x[1][0]]
    )

    ranks = contribs.reduceByKey(add).rightOuterJoin(all_nodes_pair).mapValues(
        lambda x, base=base_probability: (x[0] if x[0] is not None else 0.0) * BETA + base
    )

top_cameras = ranks.takeOrdered(10, key=lambda x: -x[1])

print("="*40)
print(f"Top 10 Cameras by PageRank (Beta={BETA})")
print("="*40)
for rank, (cam, score) in enumerate(top_cameras, 1):
    print(f"{rank}. Camera {cam:<10} : {score:.6f}")
print("="*40 + "\n")

In [ ]:
import sys
import os
from operator import add
import networkx as nx
import matplotlib.pyplot as plt
from pyspark import SparkContext, SparkConf

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

conf = SparkConf().setAppName("TrafficVis").setMaster("local[2]")

try:
    sc.stop()
except:
    pass

sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

INPUT_FILE = "file:///home/reza/HW4/camera_passes.csv"

def parse_traffic_line(line):
    try:
        parts = line.split(',')
        if parts[0] == "RecordID" or parts[1] == "CameraID":
            return None
        camera_id = parts[1].strip()
        plate_number = parts[2].strip()
        record_time = parts[3].strip()
        if not camera_id: return None
        return (plate_number, (record_time, camera_id))
    except:
        return None

raw_rdd = sc.textFile(INPUT_FILE)
parsed_rdd = raw_rdd.map(parse_traffic_line).filter(lambda x: x is not None)

def create_transitions(data):
    plate, sightings = data
    sorted_sightings = sorted(sightings, key=lambda x: x[0])
    links = []
    for i in range(len(sorted_sightings) - 1):
        src = sorted_sightings[i][1]
        dst = sorted_sightings[i+1][1]
        if src != dst:
            links.append((src, dst))
    return links

edges = parsed_rdd.groupByKey().flatMap(create_transitions)

edge_weights = edges.map(lambda x: (x, 1)).reduceByKey(add)

edges.cache()
edge_weights.cache()

links = edges.distinct().groupByKey().mapValues(list)
all_nodes = edges.flatMap(lambda x: [x[0], x[1]]).distinct()
total_nodes_count = all_nodes.count()
if total_nodes_count == 0:
    raise ValueError("The transition graph contains no nodes")
dangling_nodes = all_nodes.subtract(links.map(lambda x: x[0]))
full_links = links.union(dangling_nodes.map(lambda x: (x, [])))
ranks = all_nodes.map(lambda x: (x, 1.0 / total_nodes_count))
all_nodes_pair = all_nodes.map(lambda x: (x, None))

ITERATIONS = 10
BETA = 0.85

for i in range(ITERATIONS):
    dangling_mass = full_links.join(ranks).filter(lambda x: len(x[1][0]) == 0).map(lambda x: x[1][1]).sum()
    base_probability = ((1 - BETA) + BETA * dangling_mass) / total_nodes_count
    contribs = full_links.join(ranks).flatMap(
        lambda x: [(dest, x[1][1] / len(x[1][0])) for dest in x[1][0]]
    )
    ranks = contribs.reduceByKey(add).rightOuterJoin(all_nodes_pair).mapValues(
        lambda x, base=base_probability: (x[0] if x[0] is not None else 0.0) * BETA + base
    )

TOP_K = 15
top_nodes_list = ranks.takeOrdered(TOP_K, key=lambda x: -x[1])
top_node_ids = set([x[0] for x in top_nodes_list])

top_nodes_broadcast = sc.broadcast(top_node_ids)

subgraph_edges = edge_weights.filter(
    lambda x: x[0][0] in top_nodes_broadcast.value and x[0][1] in top_nodes_broadcast.value
).collect()

G = nx.DiGraph()

for (src, dst), weight in subgraph_edges:
    G.add_edge(src, dst, weight=weight)

plt.figure(figsize=(12, 10))
pos = nx.spring_layout(G, k=2.0, iterations=50) 

weights = [G[u][v]['weight'] for u, v in G.edges()]

max_weight = max(weights) if weights else 1
normalized_widths = [(w / max_weight) * 5 + 1 for w in weights]

node_sizes = [3000 for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='skyblue', edgecolors='black')
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

nx.draw_networkx_edges(G, pos, 
                       width=normalized_widths, 
                       edge_color='gray', 
                       arrowsize=20, 
                       connectionstyle="arc3,rad=0.1")

edge_labels = {(u, v): d['weight'] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

plt.title(f"Traffic Subgraph: Top {TOP_K} PageRank Cameras", fontsize=16)
plt.axis('off')

output_path = "/home/reza/HW4/traffic_graph_viz.png"
plt.savefig(output_path, format="PNG", dpi=300, bbox_inches='tight')
print(f"Graph visualization saved to: {output_path}")